# Silver Layer: Airline Table Cleaning & Validation

Cleanses `skyledger_airlines.bronze.airline` into `skyledger_airlines.silver.airline`.
Applies primary key validation (§3.1) and referential integrity checks (§3.4).
Invalid records are quarantined in `silver.airline_quarantine` with comma-separated rejection reasons.
Includes Bronze-Silver-Quarantine reconciliation and a data-quality report.

**Relationships:** `airline` → `flight_booking` (carrier operating each segment).
`nation` → `airline` (airline geography — airline nationality). FK `al_nationkey` validated against `silver.nation`.

In [0]:
%sql
select * from skyledger_airlines.bronze.airline

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql.functions import col, when, lit, count as _count, concat_ws
from pyspark.sql.window import Window

bronze_airline = spark.table("skyledger_airlines.bronze.airline")

# Drop metadata columns: _ingestion_timestamp, _source_file, _pipeline_run_id
# Cast key columns to INT, rating to DOUBLE for numeric handling
airline_typed = bronze_airline.select(
    col("al_airlineid").cast("int").alias("al_airlineid"),
    col("al_name").cast("string").alias("al_name"),
    col("al_address").cast("string").alias("al_address"),
    col("al_nationkey").cast("int").alias("al_nationkey"),
    col("al_phone").cast("string").alias("al_phone"),
    col("al_rating").cast("double").alias("al_rating"),
    col("al_comment").cast("string").alias("al_comment"),
)

bronze_count = airline_typed.count()
print(f"Bronze airline rows read: {bronze_count}")
display(airline_typed)

In [0]:
# ── Step 2: Primary Key Validation ──
# Rule: al_airlineid must be NOT NULL and UNIQUE.
# Failure codes: NULL_PK, DUPLICATE_PK

from pyspark.sql.functions import row_number

airline_flagged = airline_typed.withColumn(
    "is_null_pk",
    when(col("al_airlineid").isNull(), lit(True)).otherwise(lit(False)),
)

# Window for dup detection and selection
pk_window = Window.partitionBy("al_airlineid").orderBy(lit(1))

# Assign row_number to keep only one per PK
airline_flagged = airline_flagged.withColumn(
    "_pk_count", _count("*").over(Window.partitionBy("al_airlineid"))
).withColumn(
    "is_duplicate_pk",
    when((col("al_airlineid").isNotNull()) & (col("_pk_count") > 1), lit(True)).otherwise(lit(False)),
).withColumn(
    "_pk_row_number", row_number().over(pk_window)
)

# Mark as duplicate except first valid row (null PK and duplicate PK get flagged as before)
airline_flagged = airline_flagged.withColumn(
    "is_duplicate_pk",
    when((col("is_duplicate_pk") == True) & (col("_pk_row_number") > 1), lit(True)).otherwise(col("is_duplicate_pk")),
)

null_pk_count = airline_flagged.filter(col("is_null_pk")).count()
duplicate_pk_count = airline_flagged.filter((col("is_duplicate_pk") == True)).count()

print(f"NULL_PK failures:      {null_pk_count}")
print(f"DUPLICATE_PK failures: {duplicate_pk_count}")

In [0]:
# ── Step 3: Referential Integrity Validation ──
# Rule: al_nationkey must resolve to a valid n_nationkey in silver.nation.
# Failure code: INVALID_FK_NATION
#
# Parent-before-child order: nation is the parent of airline.
# Uses silver.nation (already validated) as the reference.

valid_nation_keys = (
    spark.table("skyledger_airlines.silver.nation")
    .select(col("n_nationkey").cast("int").alias("n_nationkey"))
    .filter(col("n_nationkey").isNotNull())
    .distinct()
)

# Left join: rows with no match (or null al_nationkey) get n_nationkey = null → invalid FK
airline_flagged = airline_flagged.join(
    valid_nation_keys,
    airline_flagged["al_nationkey"] == valid_nation_keys["n_nationkey"],
    "left",
).withColumn(
    "is_invalid_fk_nation",
    when(col("n_nationkey").isNull(), lit(True)).otherwise(lit(False)),
)

invalid_fk_count = airline_flagged.filter(col("is_invalid_fk_nation")).count()
print(f"INVALID_FK_NATION failures: {invalid_fk_count}")

In [0]:
# ── Step 4: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──

# Build comma-separated rejection_reason from all flag columns
airline_flagged = airline_flagged.withColumn(
    "rejection_reason",
    concat_ws(
        ", ",
        when(col("is_null_pk"), lit("NULL_PK")),
        when(col("is_duplicate_pk"), lit("DUPLICATE_PK")),
        when(col("is_invalid_fk_nation"), lit("INVALID_FK_NATION")),
    ),
)

# Clean records: passed every check (rejection_reason is empty string)
silver_airline = airline_flagged.filter(col("rejection_reason") == "").select(
    "al_airlineid", "al_name", "al_address", "al_nationkey",
    "al_phone", "al_rating", "al_comment",
)

# Quarantined records: failed one or more checks
quarantine_airline = airline_flagged.filter(col("rejection_reason") != "").select(
    "al_airlineid", "al_name", "al_address", "al_nationkey",
    "al_phone", "al_rating", "al_comment", "rejection_reason",
)

silver_count = silver_airline.count()
quarantine_count = quarantine_airline.count()

print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# Write Silver table (overwrite for idempotency — same result on every run)
(silver_airline.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.airline"))

# Write Quarantine table (overwrite for idempotency)
(quarantine_airline.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("skyledger_airlines.silver.airline_quarantine"))

print("✅ Tables written: silver.airline, silver.airline_quarantine")

In [0]:
# ── Step 5: Bronze-Silver-Quarantine Reconciliation  ──
# Bronze row count must equal Silver + Quarantine for the airline table.

bronze_total = spark.table("skyledger_airlines.bronze.airline").count()
silver_total = spark.table("skyledger_airlines.silver.airline").count()
quarantine_total = spark.table("skyledger_airlines.silver.airline_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != "
    f"Silver ({silver_total}) + Quarantine ({quarantine_total})"
)

print("✅ Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 6: Data Quality Report ──
# Documents every check executed, even when zero rows fail.

dq_report = spark.createDataFrame(
    [
        ("airline", "NULL_PK", "al_airlineid must not be null", null_pk_count, "Quarantine"),
        ("airline", "DUPLICATE_PK", "al_airlineid must be unique across all rows", duplicate_pk_count, "Quarantine"),
        ("airline", "INVALID_FK_NATION", "al_nationkey must reference a valid n_nationkey in silver.nation", invalid_fk_count, "Quarantine"),
    ],
    ["table", "rule_code", "check_description", "failing_rows", "action_taken"],
)

display(dq_report)

# Show quarantined records if any exist
if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table("skyledger_airlines.silver.airline_quarantine"))
else:
    print("\nNo quarantined records — all airline rows passed every check.")

# Show final Silver table
print(f"\n── Silver Airline Table ({silver_count} rows) ──")
display(spark.table("skyledger_airlines.silver.airline"))